# Updated Fast Colab Path

This notebook is the Colab feature-extraction entrypoint for the rerun. It supports the dense subset by default (`configs/exp1_under12h_dense.yaml`) and includes commands for main global features and dense patch-token features. Run it after the local Blender/post-render stage has produced `render_valid.parquet` and the repo/data tree has synced to Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Change this if your Drive checkout uses a different folder.
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/cv-project')
os.environ['CV_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ.setdefault('PYTHONPATH', str(PROJECT_ROOT))
%cd {PROJECT_ROOT}

!nvidia-smi

In [ ]:
import sys

# Install only the pieces needed for frozen feature extraction.
!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q transformers open_clip_torch pyarrow safetensors accelerate

In [ ]:
from pathlib import Path
import pandas as pd

LOCAL_PROJECT_ROOT = Path('/Users/jerry/cv-project')
PATH_COLUMNS = ['rgb_path', 'depth_path', 'normal_path', 'mask_path']

def make_colab_manifest(source: str, output: str) -> Path:
    source_path = Path(source)
    output_path = Path(output)
    if not source_path.is_file():
        raise FileNotFoundError(f'Missing {source_path}. Sync rendered data/manifests to Drive first.')
    df = pd.read_parquet(source_path)
    for column in PATH_COLUMNS:
        if column not in df.columns:
            continue
        def rebase(value):
            path = Path(str(value))
            if path.is_absolute():
                try:
                    path = path.relative_to(LOCAL_PROJECT_ROOT)
                except ValueError:
                    path = Path(*path.parts[1:])
            return str(path)
        df[column] = df[column].map(rebase)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)
    print(f'Wrote {len(df)} rows to {output_path}')
    return output_path

MAIN_MANIFEST = make_colab_manifest(
    'data/exp1_under12h/manifests/render_valid.parquet',
    'data/exp1_under12h/manifests/render_valid_colab.parquet',
)
DENSE_MANIFEST = make_colab_manifest(
    'data/exp1_under12h_dense/manifests/render_valid.parquet',
    'data/exp1_under12h_dense/manifests/render_valid_colab.parquet',
)

In [ ]:
# Main global CLS features: 8,307 renders x 3 models x 4 layers.
# This is the stage that previously took about 1.5-2 hours on Colab.
!PYTHONPATH=. python scripts/extract_exp1_features.py \
  --config configs/exp1_under12h.yaml \
  --render-manifest {MAIN_MANIFEST} \
  --feature-dir data/exp1_under12h/features \
  --models clip_vit_b16 clip_vit_l14 dinov2_vit_b \
  --layers final layer4 layer8 layer12 \
  --device cuda \
  --batch-size 64 \
  --num-workers 4 \
  --amp

In [ ]:
# Dense patch-token features: dense subset by default for speed.
# Increase layers/models only if you have storage and time for the larger ablation.
!PYTHONPATH=. python scripts/extract_exp1_patch_features.py \
  --config configs/exp1_under12h_dense.yaml \
  --render-manifest {DENSE_MANIFEST} \
  --feature-dir data/exp1_under12h_dense/features \
  --models clip_vit_b16 dinov2_vit_b \
  --layers final layer8 \
  --include-cls \
  --dtype float16 \
  --device cuda \
  --batch-size 48 \
  --num-workers 4 \
  --amp

In [ ]:
# Optional: extract dense patch features over the full 8.3k main run.
# This is much heavier than the dense subset and should be run only if needed.
RUN_FULL_DENSE_PATCHES = False
if RUN_FULL_DENSE_PATCHES:
    !PYTHONPATH=. python scripts/extract_exp1_patch_features.py \
      --config configs/exp1_under12h.yaml \
      --render-manifest {MAIN_MANIFEST} \
      --feature-dir data/exp1_under12h/features \
      --models clip_vit_b16 clip_vit_l14 dinov2_vit_b \
      --layers final layer4 layer8 layer12 \
      --include-cls \
      --dtype float16 \
      --device cuda \
      --batch-size 32 \
      --num-workers 4 \
      --amp

# Experiment 1 Under-12h Patch Feature Extraction

Run this notebook on a Google Colab GPU for the dense patch-depth sub-study after the dense render QC/post-render step has produced `data/exp1_under12h_dense/manifests/render_valid.parquet`. It caches patch-token grids for CLIP ViT-B/16 and DINOv2 ViT-B using float16 storage by default.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Update this if your repo lives elsewhere in Drive.
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/cv-project')
os.environ['CV_PROJECT_ROOT'] = str(PROJECT_ROOT)
%cd {PROJECT_ROOT}

!nvidia-smi

In [ ]:
import sys

# Install project and model dependencies. Re-running this cell is safe.
!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q transformers open_clip_torch pyarrow safetensors

In [ ]:
import pandas as pd
from pathlib import Path

# Default: dense patch-depth sub-study. To extract patches over the main run instead,
# switch these three paths to configs/exp1_under12h.yaml and data/exp1_under12h.
CONFIG = 'configs/exp1_under12h_dense.yaml'
SOURCE_RENDER_MANIFEST = Path('data/exp1_under12h_dense/manifests/render_valid.parquet')
VALID_RENDER_MANIFEST = Path('data/exp1_under12h_dense/manifests/render_valid_colab.parquet')
FEATURE_DIR = Path('data/exp1_under12h_dense/features')
PATCH_LAYERS = ['final', 'layer8']
PATCH_DTYPE = 'float16'
PATH_COLUMNS = ['rgb_path', 'depth_path', 'normal_path', 'mask_path']
LOCAL_PROJECT_ROOT = Path('/Users/jerry/cv-project')

if not SOURCE_RENDER_MANIFEST.is_file():
    raise FileNotFoundError(f'Missing {SOURCE_RENDER_MANIFEST}. Render/QC the dense sub-study first and sync data to Drive.')

manifest = pd.read_parquet(SOURCE_RENDER_MANIFEST)
for column in PATH_COLUMNS:
    if column in manifest.columns:
        def rebase_path(value):
            path = Path(str(value))
            if path.is_absolute():
                try:
                    path = path.relative_to(LOCAL_PROJECT_ROOT)
                except ValueError:
                    path = Path(*path.parts[1:])
            return str(PROJECT_ROOT / path)
        manifest[column] = manifest[column].map(rebase_path)

VALID_RENDER_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
manifest.to_parquet(VALID_RENDER_MANIFEST, index=False)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Validated dense renders: {len(manifest):,}')
print(f'Colab manifest: {VALID_RENDER_MANIFEST}')
print(manifest.groupby(['split', 'texture_condition']).size())

In [ ]:
# CLIP ViT-B/16 patch-token grids.
!PYTHONPATH=. python scripts/extract_exp1_patch_features.py \
  --config {CONFIG} \
  --render-manifest {VALID_RENDER_MANIFEST} \
  --feature-dir {FEATURE_DIR} \
  --models clip_vit_b16 \
  --layers {' '.join(PATCH_LAYERS)} \
  --device cuda \
  --batch-size 32 \
  --num-workers 4 \
  --dtype {PATCH_DTYPE} \
  --amp

In [ ]:
# DINOv2 ViT-B patch-token grids. Reduce batch size to 16 if Colab reports CUDA OOM.
!PYTHONPATH=. python scripts/extract_exp1_patch_features.py \
  --config {CONFIG} \
  --render-manifest {VALID_RENDER_MANIFEST} \
  --feature-dir {FEATURE_DIR} \
  --models dinov2_vit_b \
  --layers {' '.join(PATCH_LAYERS)} \
  --device cuda \
  --batch-size 24 \
  --num-workers 4 \
  --dtype {PATCH_DTYPE} \
  --amp

In [ ]:
# Quick patch-cache inventory for sync/checkpointing.
for path in sorted(FEATURE_DIR.glob('**/*patch*')):
    if path.is_file():
        size_gb = path.stat().st_size / 1e9
        print(f'{path}  {size_gb:.2f} GB')